# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple, Iterator, List, Iterable, Tuple

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.1711469512288635)),
 (1, np.float64(1.1711469512288635)),
 (2, np.float64(1.1711469512288635)),
 (3, np.float64(1.1711469512288635)),
 (4, np.float64(1.1711469512288635))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=REDUCE)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

16 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('it', 18)]),
 (1, [('banana', 2), ('is', 18), ('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.006734879584062381)),
   (None, np.float64(0.027378064547203818)),
   (None, np.float64(0.046481064998160515)),
   (None, np.float64(0.04798626592866684)),
   (None, np.float64(0.07312519025226383)),
   (None, np.float64(0.08779649539303358)),
   (None, np.float64(0.12631238721697713)),
   (None, np.float64(0.13466546604212115)),
   (None, np.float64(0.2024506985614638)),
   (None, np.float64(0.20741099478982505)),
   (None, np.float64(0.21682290797110992)),
   (None, np.float64(0.286091627315083)),
   (None, np.float64(0.33436943815784415)),
   (None, np.float64(0.3863988979149302)),
   (None, np.float64(0.4758171339562446))]),
 (1,
  [(None, np.float64(0.545220958691045)),
   (None, np.float64(0.6083263035892815)),
   (None, np.float64(0.6377690335978212)),
   (None, np.float64(0.6605050540736825)),
   (None, np.float64(0.6671814678709597)),
   (None, np.float64(0.6755313385323822)),
   (None, np.float64(0.6953490606228957)),
   (None, np.float64(0.6999417

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
# Входные данные
input_numbers = [23, 45, 12, 67, 34, 89, 5, 72, 18, 56, 91, 33]

def RECORDREADER():
    """Читает входные данные и возвращает пары (ключ, значение)"""
    for i, num in enumerate(input_numbers):
        yield (i, num)  # уникальный ключ для каждого числа

def MAP(_, value: int):
    """Map функция - просто передает число с ключом 'max' для группировки"""
    yield ('max', value)  # все числа идут с одним ключом

def REDUCE(key: str, values: Iterator[int]):
    """Reduce функция - находит максимум среди всех значений"""
    max_value = float('-inf')
    for v in values:
        if v > max_value:
            max_value = v
    yield (key, max_value)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('max', 91)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def MAP(_, value: int):
    """Map функция - просто передает число с ключом 'mean' для группировки"""
    yield ('mean', value)  # все числа идут с одним ключом

def REDUCE(key: str, values: Iterator[int]):
    """Reduce функция - находит среднее всех значений"""
    result = 0
    count = 0
    for v in values:
        result += v
        count += 1
    yield result/count

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[45.416666666666664]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
import itertools

def groupByKey_sort(pairs):
    pairs_sorted = sorted(pairs, key=lambda pair: pair[0])

    for key, group_iter in itertools.groupby(pairs_sorted, key=lambda pair: pair[0]):
        values = []
        for _, value in group_iter:
            values.append(value)
        yield (key, values)

pairs = [('b', 1), ('a', 2), ('b', 3), ('a', 4), ('c', 5), ('b', 6)]
print(list(groupByKey_sort(pairs)))

pairs = [(10, '10'), (10, 20), (15, 30), (10, 40), (15, 20), (20, 20), (20, 30)]
print(list(groupByKey_sort(pairs)))

[('a', [2, 4]), ('b', [1, 3, 6]), ('c', [5])]
[(10, ['10', 20, 40]), (15, [30, 20]), (20, [20, 30])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
input_numbers = [1, 2, 1, 2, 3, 4, 1, 3, 2, 4, 1, 2]

def MAP(_, value: int):
    yield (value, 1)

def REDUCE(key: int, values: Iterator[int]):
    yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (0, User(id=1, age=25, social_contacts=240, gender='female')),
 (0, User(id=2, age=25, social_contacts=500, gender='female')),
 (0, User(id=3, age=33, social_contacts=800, gender='female')),
 (1, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=4, age=215, social_contacts=24000, gender='female')),
 (1, User(id=21, age=35, social_contacts=50, gender='female')),
 (1, User(id=3, age=33, social_contacts=800, gender='female'))]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

def C(value: int):
  return value > 30

def MAP(_, row: NamedTuple):
  if C(row.age):
      yield (row, row)

def REDUCE(_, values: Iterator[int]):
  for val in values:
    yield val

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=3, age=33, social_contacts=800, gender='female')]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
S = {1, 3}  # индексы нужных атрибутов

def MAP(_, t: tuple):
    t_prime = tuple(t[i] for i in S if i < len(t))
    yield (t_prime, t_prime)

def REDUCE(key: tuple, values: Iterator[tuple]):
    yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(55, 'male'), (25, 'female'), (33, 'female')]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]
S = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=4, age=215, gender='female', social_contacts=24000),
    User(id=21, age=35, gender='female', social_contacts=50),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    for idx, record in enumerate(R):
        yield (f"R_{idx}", (0, record))

    for idx, record in enumerate(S):
        yield (f"S_{idx}", (1, record))

def MAP(record_id: str, source_and_record: tuple):
    source_id, record = source_and_record
    yield (record, source_id)

def REDUCE(key: tuple, values: Iterator[tuple]):
    yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=1, age=25, social_contacts=240, gender='female'),
 User(id=2, age=25, social_contacts=500, gender='female'),
 User(id=3, age=33, social_contacts=800, gender='female'),
 User(id=4, age=215, social_contacts=24000, gender='female'),
 User(id=21, age=35, social_contacts=50, gender='female')]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
def REDUCE(key: tuple, values: Iterator[tuple]):
    if len(values) > 1:
      yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=3, age=33, social_contacts=800, gender='female')]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
def REDUCE(key: tuple, values: Iterator[tuple]):
    if values == [1]:
      yield key

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[User(id=4, age=215, social_contacts=24000, gender='female'),
 User(id=21, age=35, social_contacts=50, gender='female')]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
R = [
    (1, "x"),
    (2, "y"),
    (3, "x"),
]

S = [
    ("x", 100),
    ("x", 200),
    ("z", 300),
]

def RECORDREADER():
  for a, b in R:
    yield ("R", (a, b))
  for b, c in S:
    yield ("S", (b, c))

def MAP(key: str, values: Iterator[tuple]):
  if key == "R":
    a, b = values
    yield (b, (key, a))
  else:
    b, c = values
    yield (b, (key, c))

def REDUCE(b, values: Iterator[tuple]):
  r_list = []
  s_list = []

  for key, val in values:
    if key == "R":
      r_list.append(val)
    else:
      s_list.append(val)

  for a in r_list:
    for c in s_list:
      yield (a, b, c)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(1, 'x', 100), (1, 'x', 200), (3, 'x', 100), (3, 'x', 200)]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=1, age=25, gender='female', social_contacts=500),
    User(id=0, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    for t in R:
        yield (t, t)

def MAP(_, t: Iterator[tuple]):
    a, b = t.id, t.social_contacts
    yield (a, b)

def REDUCE(a, values: Iterator[tuple]):
    sum = 0
    for val in values:
      sum += val
    yield (a, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 820), (1, 740)]

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all recordreader tasks

def MAP(i:int, mat_val:int, vec_val:int):
  yield (i, mat_val*vec_val)

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield (i, mat[i,j], vec[j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.533822302314017)),
 (1, np.float64(2.533822302314017)),
 (2, np.float64(2.533822302314017)),
 (3, np.float64(2.533822302314017)),
 (4, np.float64(2.533822302314017))]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1

  for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  s = 0.0
  for v in values:
    s += v
  yield (key, s)

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
M = np.random.randint(1, 10, (3, 4))
N = np.random.randint(1, 10, (4, 3))

def RECORDREADER():
    for i in range(3):
        for j in range(4):
            yield ("M", (i, j, M[i, j]))

    for j in range(4):
        for k in range(3):
            yield ("N", (j, k, N[j, k]))

def MAP(matrix_type: str, matrix_data: tuple):
    if matrix_type == 'M':
        i, j, v = matrix_data
        yield (j, ('M', i, v))

    else:
        j, k, w = matrix_data
        yield (j, ('N', k, w))

def REDUCE(j: int, values: Iterator[tuple]):

    m_elements = []
    n_elements = []

    for val in values:
        if val[0] == 'M':
            _, i, v = val
            m_elements.append((i, v))
        else:
            _, k, w = val
            n_elements.append((k, w))

    for (i, v) in m_elements:
        for (k, w) in n_elements:
            product = v * w
            yield ((i, k), product)

def REDUCE_FINAL(pair: tuple, products: Iterator[int]):
    i, k = pair
    total = sum(products)
    yield ((i, k), total)

def MapReduceTwoSteps(RECORDREADER, MAP, REDUCE, REDUCE_FINAL):
  return flatten(map(lambda x: REDUCE_FINAL(*x), groupbykey(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))))


reference_solution = np.matmul(M, N)
solution = MapReduceTwoSteps(RECORDREADER, MAP, REDUCE, REDUCE_FINAL)
np.allclose(reference_solution, asmatrix(solution))

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
maps = 3
reducers = 4

def chunkify(items: List, n_chunks: int) -> List[List]:
    n_chunks = max(1, int(n_chunks))
    out = [[] for _ in range(n_chunks)]
    for idx, x in enumerate(items):
        out[idx % n_chunks].append(x)
    return out

def RECORDREADER_M() -> Iterable[Tuple[Tuple[str, int, int], float]]:
    for i in range(3):
        for j in range(4):
            yield (("M", i, j), float(M[i, j]))

def RECORDREADER_N() -> Iterable[Tuple[Tuple[str, int, int], float]]:
    for j in range(4):
        for k in range(3):
            yield (("N", j, k), float(N[j, k]))

def INPUTFORMAT1():
    a_items = list(RECORDREADER_M())
    b_items = list(RECORDREADER_N())

    a_maps = max(1, maps // 2)
    b_maps = max(1, maps - a_maps)

    a_chunks = chunkify(a_items, a_maps)
    b_chunks = chunkify(b_items, b_maps)

    def rr_from_chunk(chunk):
        for k1, v1 in chunk:
            yield (k1, v1)


    for ch in a_chunks:
        yield rr_from_chunk(ch)
    for ch in b_chunks:
        yield rr_from_chunk(ch)

def MAP1(k1, v1):
    tag = k1[0]
    if tag == "M":
        _, i, j = k1
        yield (j, ("M", i, float(v1)))
    else:
        _, j, k = k1
        yield (j, ("N", k, float(v1)))

def REDUCE1(j: int, vals: Iterator[tuple]):
    left = []
    right = []

    for rec in vals:
        if rec[0] == "M":
            _, i, aij = rec
            left.append((i, aij))
        else:
            _, k, bjk = rec
            right.append((k, bjk))

    for i, aij in left:
        for k, bjk in right:
            yield ((i, k), aij * bjk)

stage1 = MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1)
stage1 = [(pid, list(part)) for (pid, part) in stage1]
partials = [kv for (_, part) in stage1 for kv in part]

def INPUTFORMAT2():
    chunks = chunkify(partials, maps)

    def rr_from_chunk(chunk):
        for (ik, val) in chunk:
            yield (ik, float(val))

    for ch in chunks:
        yield rr_from_chunk(ch)

def MAP2(ik, val: float):
    yield (ik, float(val))

def REDUCE2(ik, vals: Iterator[float]):
    s = 0.0
    for x in vals:
        s += x
    yield (ik, s)

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2 = [(pid, list(part)) for (pid, part) in stage2]
out = [kv for (_, part) in stage2 for kv in part]

def to_dense(pairs, I: int, K: int):
    mat = np.zeros((I, K), dtype=float)
    for (i, k), v in pairs:
        mat[i, k] = v
    return mat

solution = to_dense(out, 3, 3)
np.set_printoptions(precision=8, suppress=False)
print(repr(solution))
print(repr(reference_solution))
np.allclose(reference_solution, solution)

24 key-value pairs were sent over a network.
18 key-value pairs were sent over a network.
array([[ 90., 122.,  96.],
       [ 81., 100.,  95.],
       [ 95., 119., 108.]])
array([[ 90, 122,  96],
       [ 81, 100,  95],
       [ 95, 119, 108]])


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
np.random.seed(4)

I, J, K = 4, 5, 3
A = np.random.randn(I, J).astype(float)
B = np.random.randn(J, K).astype(float)
reference = A @ B

maps = 6
reducers = 3

A_entries = [(("A", i, j), float(A[i, j])) for i in range(I) for j in range(J)]
B_entries = [(("B", j, k), float(B[j, k])) for j in range(J) for k in range(K)]

def random_partition_full(entries: List[Tuple[tuple, float]], n_parts: int, seed: int):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(entries))
    rng.shuffle(idx)
    parts = np.array_split(idx, n_parts)
    return [[entries[i] for i in part] for part in parts]

A_splits = random_partition_full(A_entries, n_parts=3, seed=10)
B_splits = random_partition_full(B_entries, n_parts=3, seed=20)

def INPUTFORMAT1():
    def RR(split):
        for kv in split:
            yield kv
    for sp in A_splits:
        yield RR(sp)
    for sp in B_splits:
        yield RR(sp)

def MAP1(k1, v1):
    tag = k1[0]
    if tag == "A":
        _, i, j = k1
        yield (j, ("A", i, float(v1)))
    else:
        _, j, k = k1
        yield (j, ("B", k, float(v1)))

def REDUCE1(j: int, tagged_vals: Iterator[tuple]):
    left = []
    right = []
    for rec in tagged_vals:
        if rec[0] == "A":
            _, i, aij = rec
            left.append((i, aij))
        else:
            _, k, bjk = rec
            right.append((k, bjk))

    for i, aij in left:
        for k, bjk in right:
            yield ((i, k), aij * bjk)

stage1 = MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1, COMBINER=None)
stage1 = [(pid, list(part)) for (pid, part) in stage1]
partials = [kv for (_, part) in stage1 for kv in part]

print("CASE1 partials:", len(partials), "expected:", I * J * K)

maps = 4
reducers = 3

def split_random(items, n_parts, seed):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)
    parts = np.array_split(idx, n_parts)

    def RR(index_chunk):
        for t in index_chunk:
            yield items[t]
    for part in parts:
        yield RR(part)

def INPUTFORMAT2():
    return split_random(partials, n_parts=maps, seed=30)

def MAP2(key, val):
    yield (key, float(val))

def REDUCE2(key, vals: Iterator[float]):
    s = 0.0
    for x in vals:
        s += x
    yield (key, s)

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2 = [(pid, list(part)) for (pid, part) in stage2]
out = [kv for (_, part) in stage2 for kv in part]

def asmatrix(pairs, I, K):
    mat = np.zeros((I, K), dtype=float)
    for ((i, k), v) in pairs:
        mat[i, k] = v
    return mat

solution = asmatrix(out, I, K)

print("Matches numpy (full cover, random RR):", np.allclose(reference, solution))
np.set_printoptions(precision=8, suppress=False)
print(repr(solution))
print(repr(reference))

maps = 6
reducers = 3

def random_subset(entries, frac, seed):
    rng = np.random.default_rng(seed)
    n = len(entries)
    m = int(np.floor(frac * n))
    idx = rng.choice(n, size=m, replace=False)
    return [entries[i] for i in idx]

A_subset = random_subset(A_entries, frac=0.7, seed=111)
B_subset = random_subset(B_entries, frac=0.7, seed=222)

A_splits2 = random_partition_full(A_subset, n_parts=3, seed=333)
B_splits2 = random_partition_full(B_subset, n_parts=3, seed=444)


def INPUTFORMAT1_bad():
    def RR(split):
        for kv in split:
            yield kv
    for sp in A_splits2:
        yield RR(sp)
    for sp in B_splits2:
        yield RR(sp)

stage1_bad = MapReduceDistributed(INPUTFORMAT1_bad, MAP1, REDUCE1, COMBINER=None)
stage1_bad = [(pid, list(part)) for (pid, part) in stage1_bad]
partials_bad = [kv for (_, part) in stage1_bad for kv in part]
print("\nCASE2 partials (subset):", len(partials_bad), "expected(full):", I * J * K)

maps = 4
reducers = 3

def INPUTFORMAT2_bad():
    return split_random(partials_bad, n_parts=maps, seed=555)

stage2_bad = MapReduceDistributed(INPUTFORMAT2_bad, MAP2, REDUCE2, COMBINER=REDUCE2)
stage2_bad = [(pid, list(part)) for (pid, part) in stage2_bad]
out_bad = [kv for (_, part) in stage2_bad for kv in part]
solution_bad = asmatrix(out_bad, I, K)

print("Matches numpy (random SUBSET, losses):", np.allclose(reference, solution_bad))
print("Max abs diff:", float(np.max(np.abs(reference - solution_bad))))

35 key-value pairs were sent over a network.
CASE1 partials: 60 expected: 60
36 key-value pairs were sent over a network.
Matches numpy (full cover, random RR): True
array([[-3.30359593,  2.7572802 , -0.5851318 ],
       [ 2.8464065 ,  0.29702452,  1.91922881],
       [-0.29012046,  1.87650961,  2.21872797],
       [ 2.02448215, -3.63157051,  2.6886677 ]])
array([[-3.30359593,  2.7572802 , -0.5851318 ],
       [ 2.8464065 ,  0.29702452,  1.91922881],
       [-0.29012046,  1.87650961,  2.21872797],
       [ 2.02448215, -3.63157051,  2.6886677 ]])
24 key-value pairs were sent over a network.

CASE2 partials (subset): 26 expected(full): 60
24 key-value pairs were sent over a network.
Matches numpy (random SUBSET, losses): False
Max abs diff: 2.5659071726590548
